# Carga de Datos - Modelo Predictivo de Riesgo Crediticio

**Proyecto Integrador M5** - Daniel Palmera

Este notebook simula el proceso de carga de datos desde el Data Warehouse / Data Lake de la empresa. En un entorno productivo, la informacion llegaria materializada en una tabla del DWH mediante un proceso de ETL independiente. Para efectos de este proyecto, se utiliza un dataset de ejemplo no productivo en formato .xlsx, entregado por la organizacion.

**Objetivo de este notebook:**
- Cargar el archivo fuente Base_de_datos.xlsx.
- Validar la carga (dimensiones, columnas, tipos de datos base).
- Persistir una copia intermedia en formato .csv para las siguientes etapas del pipeline (EDA, feature engineering).

In [1]:
import pandas as pd
import numpy as np
import json
import os

with open('config.json') as f:
    config = json.load(f)

print(f"Proyecto: {config['project_name']}")
print(f"Codigo: {config['project_code']}")
print(f"Variable objetivo: {config['target_variable']}")


Proyecto: Modelo Predictivo de Riesgo Crediticio
Codigo: pi-m5-credit-risk
Variable objetivo: Pago_atiempo


## 1. Carga del archivo fuente

In [2]:
RAW_PATH = "../data/raw/Base_de_datos.xlsx"

df = pd.read_excel(RAW_PATH)

print(f"Filas: {df.shape[0]:,}")
print(f"Columnas: {df.shape[1]}")
df.head()


Filas: 10,763
Columnas: 23


,tipo_credito,fecha_prestamo,capital_prestado,plazo_meses,edad_cliente,tipo_laboral,salario_cliente,total_otros_prestamos,cuota_pactada,puntaje,...,saldo_mora,saldo_total,saldo_principal,saldo_mora_codeudor,creditos_sectorFinanciero,creditos_sectorCooperativo,creditos_sectorReal,promedio_ingresos_datacredito,tendencia_ingresos,Pago_atiempo
0,7,2024-12-21 11:31:35,3692160.0,10,42,Independiente,8000000,2500000,341296,88.768094,...,0.0,51258.0,51258.0,0.0,5,0,0,908526.0,Estable,1
1,4,2025-04-22 09:47:35,840000.0,6,60,Empleado,3000000,2000000,124876,95.227787,...,0.0,8673.0,8673.0,0.0,0,0,2,939017.0,Creciente,1
2,9,2026-01-08 12:22:40,5974028.4,10,36,Independiente,4036000,829000,529554,47.613894,...,0.0,18702.0,18702.0,0.0,3,0,0,NaN,NaN,0
3,4,2025-08-04 12:04:10,1671240.0,6,48,Empleado,1524547,498000,252420,95.227787,...,0.0,15782.0,15782.0,0.0,3,0,0,1536193.0,Creciente,1
4,9,2025-04-26 11:24:26,2781636.0,11,44,Empleado,5000000,4000000,217037,95.227787,...,0.0,204804.0,204804.0,0.0,3,0,1,933473.0,Creciente,1


## 2. Validacion estructural

In [3]:
print("=== Tipos de datos ===")
print(df.dtypes)


=== Tipos de datos ===
tipo_credito                              int64
fecha_prestamo                   datetime64[ns]
capital_prestado                        float64
plazo_meses                               int64
edad_cliente                              int64
tipo_laboral                             object
salario_cliente                           int64
total_otros_prestamos                     int64
cuota_pactada                             int64
puntaje                                 float64
puntaje_datacredito                     float64
cant_creditosvigentes                     int64
huella_consulta                           int64
saldo_mora                              float64
saldo_total                             float64
saldo_principal                         float64
saldo_mora_codeudor                     float64
creditos_sectorFinanciero                 int64
creditos_sectorCooperativo                int64
creditos_sectorReal                       int64
promedio_ingresos

In [4]:
print("=== Valores nulos por columna ===")
nulls = df.isnull().sum()
nulls_pct = (nulls / len(df) * 100).round(2)
resumen_nulls = pd.DataFrame({'nulos': nulls, 'pct': nulls_pct})
resumen_nulls[resumen_nulls['nulos'] > 0].sort_values('nulos', ascending=False)


=== Valores nulos por columna ===


,nulos,pct
tendencia_ingresos,2932,27.24
promedio_ingresos_datacredito,2930,27.22
saldo_mora_codeudor,590,5.48
saldo_principal,405,3.76
saldo_mora,156,1.45
saldo_total,156,1.45
puntaje_datacredito,6,0.06


In [5]:
print("=== Duplicados ===")
print(f"Filas duplicadas: {df.duplicated().sum()}")
print()
print("=== Distribucion de la variable objetivo ===")
print(df[config['target_variable']].value_counts())
print((df[config['target_variable']].value_counts(normalize=True) * 100).round(2))


=== Duplicados ===
Filas duplicadas: 0

=== Distribucion de la variable objetivo ===
Pago_atiempo
1    10252
0      511
Name: count, dtype: int64
Pago_atiempo
1    95.25
0     4.75
Name: proportion, dtype: float64


**Observacion inicial:** la variable objetivo Pago_atiempo presenta un desbalance severo (~95% clase 1 vs ~5% clase 0). Este hallazgo se profundiza en el notebook de EDA y determina la estrategia de modelado (metricas sensibles a desbalance, uso de class_weight, umbral de decision ajustado).

## 3. Persistencia intermedia

Se guarda una copia en .csv en data/processed/ para que las siguientes etapas del pipeline (EDA, ingenieria de caracteristicas) no dependan del archivo Excel original.

In [6]:
os.makedirs("../data/processed", exist_ok=True)
OUT_PATH = "../data/processed/dataset_cargado.csv"
df.to_csv(OUT_PATH, index=False)
print(f"Dataset guardado en: {OUT_PATH}")
print(f"Filas: {len(df):,} | Columnas: {df.shape[1]}")


Dataset guardado en: ../data/processed/dataset_cargado.csv
Filas: 10,763 | Columnas: 23


## 4. Nota metodologica

Este notebook (cargar_datos.ipynb) es un componente de arranque del pipeline y **no representa el flujo productivo real**. En produccion, los datos de creditos historicos residirian en el Data Warehouse corporativo y este paso seria reemplazado por una consulta SQL o una lectura desde el Data Lake, materializada por un proceso ETL externo al equipo de Ciencia de Datos.

El siguiente paso del pipeline es comprension_eda.ipynb, donde se realiza el analisis exploratorio completo.